In [6]:
from google.colab import files

uploaded = files.upload()

Saving neo_asteroid_dataset.csv to neo_asteroid_dataset.csv


In [7]:
import pandas as pd

CSV_PATH = "/content/neo_asteroid_dataset.csv"
df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nPHA counts:")
print(df["pha"].value_counts(dropna=False))

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

Dataset shape: (42815, 21)

Columns:
['spkid', 'pdes', 'full_name', 'kind', 'neo', 'pha', 'e', 'a', 'q', 'i', 'om', 'w', 'moid', 'H', 'diameter', 'albedo', 'first_obs', 'last_obs', 'data_arc', 'n_obs_used', 'condition_code']

PHA counts:
pha
N      40130
Y       2550
NaN      135
Name: count, dtype: int64

Missing values:
albedo            41611
diameter          41569
data_arc            437
pha                 135
moid                135
H                     9
condition_code        2
neo                   0
pdes                  0
spkid                 0
kind                  0
full_name             0
a                     0
w                     0
om                    0
i                     0
q                     0
e                     0
first_obs             0
last_obs              0
n_obs_used            0
dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

FEATURES = ["a", "e", "i", "om", "w", "diameter"]
TARGET = "pha"

data = df.dropna(subset=[TARGET]).copy()

# Convert target labels to 0 and 1 if they are Y/N.
data[TARGET] = data[TARGET].astype(str).str.strip().str.upper()
data = data[data[TARGET].isin(["Y", "N"])].copy()
data[TARGET] = data[TARGET].map({"N": 0, "Y": 1})

# Ensure feature columns are numeric.
for col in FEATURES:
    data[col] = pd.to_numeric(data[col], errors="coerce")

X = data[FEATURES]
y = data[TARGET].astype(int)

print("Rows retained:", len(data))
print("Features:", FEATURES)
print("\nTarget distribution:")
print(y.value_counts())
print("\nPHA prevalence:", round(y.mean() * 100, 2), "%")

Rows retained: 42680
Features: ['a', 'e', 'i', 'om', 'w', 'diameter']

Target distribution:
pha
0    40130
1     2550
Name: count, dtype: int64

PHA prevalence: 5.97 %


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining PHA prevalence:", round(y_train.mean() * 100, 2), "%")
print("Testing PHA prevalence:", round(y_test.mean() * 100, 2), "%")

Training rows: 34144
Testing rows: 8536

Training PHA prevalence: 5.97 %
Testing PHA prevalence: 5.97 %


In [11]:
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Median imputation + missingness indicators.
# keep_empty_features prevents all-missing training columns from disappearing.
imputer = SimpleImputer(
    strategy="median",
    add_indicator=True,
    keep_empty_features=True
)

model = Pipeline([
    ("imputer", imputer),
    ("classifier", HistGradientBoostingClassifier(
        learning_rate=0.1,
        max_iter=100,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("METHOD 3: MEDIAN IMPUTATION + MISSINGNESS INDICATORS")
print("-" * 55)
print("Average precision (PR-AUC):",
      round(average_precision_score(y_test, y_prob), 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_test, y_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Non-PHA", "PHA"],
    zero_division=0
))

METHOD 3: MEDIAN IMPUTATION + MISSINGNESS INDICATORS
-------------------------------------------------------
Average precision (PR-AUC): 0.2574
Precision: 0.1527
Recall: 0.7137
F1-score: 0.2516
ROC-AUC: 0.8131

Confusion matrix:
[[6007 2019]
 [ 146  364]]

Classification report:
              precision    recall  f1-score   support

     Non-PHA       0.98      0.75      0.85      8026
         PHA       0.15      0.71      0.25       510

    accuracy                           0.75      8536
   macro avg       0.56      0.73      0.55      8536
weighted avg       0.93      0.75      0.81      8536



In [12]:
results_method_3 = {
    "Method": "Median + Missingness Indicators",
    "Average Precision": average_precision_score(y_test, y_prob),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

results_method_3_df = pd.DataFrame([results_method_3])
results_method_3_df

,Method,Average Precision,Precision,Recall,F1,ROC-AUC
0,Median + Missingness Indicators,0.257396,0.152749,0.713725,0.251642,0.813102
